In [1]:
import os
import sys
import ctypes
import resource

# =============================================================================
# 0. THE CURE: PRE-IMPORT CUDA ENVIRONMENT HACK
# =============================================================================
oscar_cuda_path = "/oscar/rt/9.6/25/spack/x86_64_v3/cuda-12.9.0-cinrl2oeqemd3szbcakkugp2vtk2fh5t"
nvvm_lib_dir = os.path.join(oscar_cuda_path, "nvvm", "lib64")
nvrtc_lib_dir = os.path.join(oscar_cuda_path, "targets", "x86_64-linux", "lib")
standard_lib_dir = os.path.join(oscar_cuda_path, "lib64")

os.environ['CUDA_HOME'] = oscar_cuda_path
os.environ['CPATH'] = os.path.join(oscar_cuda_path, 'include')
os.environ['PATH'] = os.path.join(oscar_cuda_path, 'bin') + ":" + os.environ.get('PATH', '')
os.environ['NUMBA_CUDA_DRIVER'] = "/lib64/libcuda.so"
os.environ['LD_LIBRARY_PATH'] = f"{nvvm_lib_dir}:{nvrtc_lib_dir}:{standard_lib_dir}:/lib64:" + os.environ.get('LD_LIBRARY_PATH', '')
os.environ["IPIE_USE_GPU"] = "1"

try:
    ctypes.CDLL(os.path.join(nvvm_lib_dir, "libnvvm.so"), mode=ctypes.RTLD_GLOBAL)
    print("libnvvm.so successfully injected into process memory.")
except Exception as e:
    print(f"Manual injection failed: {e}")

# =============================================================================
# 1. IMPORTS
# =============================================================================
import joblib
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
import time
import json

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"TF memory growth enabled on {len(gpus)} GPUs.")
    except RuntimeError as e:
        print(e)

from pyscf import gto, scf, lib
from ipie.utils.from_pyscf import gen_ipie_input_from_pyscf_chk
from ipie.qmc.afqmc import AFQMC
from ipie.utils.mpi import MPIHandler
import ipie.estimators.local_energy_sd
from ipie.analysis.autocorr import reblock_by_autocorr
from ipie.analysis.extraction import extract_observable

try:
    import cupy as cp
    has_cupy = True
except ImportError:
    cp = None
    has_cupy = False

# =============================================================================
# 2. CONFIGURATION
# =============================================================================
N_ATOMS = 70
TEST_SEED = 999 
SYSTEM_NAME = f"H{N_ATOMS}"
WALKERS = 2048
NUM_BLOCKS = 50
STEPS_PER_BLOCK = 1
DEPLOY_DIR = "deployment_objects"
MODEL_PATH = os.path.join(DEPLOY_DIR, f"NN_{SYSTEM_NAME}_DeltaHF.keras")

@tf.keras.utils.register_keras_serializable()
def log_cosh_loss(y_true, y_pred):
    y_true = tf.cast(tf.reshape(y_true, tf.shape(y_pred)), dtype=y_pred.dtype)
    return tf.reduce_mean(tf.math.log(tf.math.cosh(y_pred - y_true)))

# =============================================================================
# 3. HELPER FUNCTIONS
# =============================================================================
def get_dynamic_operators(mol):
    S = mol.intor('int1e_ovlp')
    h_core_ao = mol.intor('int1e_nuc') + mol.intor('int1e_kin')
    e, v = np.linalg.eigh(S)
    mask = e > 1e-15
    S_inv_sqrt = v[:, mask] @ np.diag(e[mask]**(-0.5)) @ v[:, mask].T
    S_sqrt = v[:, mask] @ np.diag(e[mask]**(0.5)) @ v[:, mask].T
    h_core_lowdin = S_inv_sqrt.T @ h_core_ao @ S_inv_sqrt
    return S_inv_sqrt, S_sqrt, h_core_lowdin, S

def estimate_nn_flops(model):
    """Dynamically estimates FLOPs for one inference pass of a single walker."""
    flops = 0
    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.Dense):
            weights = layer.get_weights()
            if len(weights) > 0:
                kernel = weights[0]
                in_dim = kernel.shape[0]
                out_dim = kernel.shape[1]
                flops += 2 * in_dim * out_dim
                if len(weights) > 1:
                    flops += out_dim
    return flops

# =============================================================================
# 4. THE ML PROXY (Fully GPU-Accelerated)
# =============================================================================
def create_ml_local_energy_patch(
    ml_model, X_scaler, y_scaler, 
    P_hf_ref, E_hf_ref, S_sqrt, h_core_dyn, C_a, 
    use_gpu, n_atoms
):
    xp = cp if use_gpu else np

    P_hf_ref_xp = xp.asarray(P_hf_ref)
    S_sqrt_xp = xp.asarray(S_sqrt)
    h_core_dyn_xp = xp.asarray(h_core_dyn)
    C_a_xp = xp.asarray(C_a)

    @tf.function(reduce_retracing=True)
    def fast_predict(inputs):
        return ml_model(inputs, training=False)

    call_counter = {"count": 0}

    def local_energy_single_det_uhf(system, hamiltonian, walkers, trial):
        if call_counter["count"] < 1:
            device_str = "GPU (CuPy/TF)" if use_gpu else "CPU (NumPy/TF)"
            print(f"\n[ML-PROXY] Intercepted call. Algebra executing on: {device_str}.")
            call_counter["count"] += 1

        nwalkers = walkers.nwalkers
        nalpha = trial.nalpha
        
        phi_a = walkers.phia if hasattr(walkers, 'phia') else walkers.phi[:, :, :nalpha]
        phi_b = walkers.phib if hasattr(walkers, 'phib') else walkers.phi[:, :, nalpha:]
        
        Psi_T_a = xp.asarray(trial.psi[:, :nalpha])
        Psi_T_b = xp.asarray(trial.psi[:, nalpha:])
        phi_a = xp.asarray(phi_a)
        phi_b = xp.asarray(phi_b)

        O_a = xp.einsum('ui, wuj -> wij', Psi_T_a.conj(), phi_a)
        O_b = xp.einsum('ui, wuj -> wij', Psi_T_b.conj(), phi_b)
        invO_a = xp.linalg.inv(O_a)
        invO_b = xp.linalg.inv(O_b)
        
        G_mo_a = xp.einsum('wvi, wiu -> wvu', phi_a, xp.einsum('wij, ju -> wiu', invO_a, Psi_T_a.conj().T))
        G_mo_b = xp.einsum('wvi, wiu -> wvu', phi_b, xp.einsum('wij, ju -> wiu', invO_b, Psi_T_b.conj().T))
        
        P_ao = (xp.einsum("qi, wij, pj -> wqp", C_a_xp, G_mo_a, C_a_xp.conj()) + 
                xp.einsum("qi, wij, pj -> wqp", C_a_xp, G_mo_b, C_a_xp.conj()))

        P_lowdin = xp.einsum('ai, wib, bj -> waj', S_sqrt_xp, P_ao, S_sqrt_xp)
        delta_P = P_lowdin - P_hf_ref_xp
        E_1B_delta = xp.einsum('ij, wji -> w', h_core_dyn_xp, delta_P).real

        delta_P = cp.asnumpy(delta_P) if use_gpu else delta_P
        E_1B_delta = cp.asnumpy(E_1B_delta) if use_gpu else E_1B_delta

        X_input = np.stack([np.real(delta_P), np.imag(delta_P)], axis=-1).astype(np.float32)
        X_flat = X_input.reshape(nwalkers, -1)
        X_scaled = X_scaler.transform(X_flat).reshape(nwalkers, n_atoms, n_atoms, 2)

        preds_scaled = fast_predict(X_scaled).numpy()
        E_corr_delta = y_scaler.inverse_transform(preds_scaled).flatten()

        energy_out = xp.zeros((nwalkers, 3), dtype=xp.complex128)
        energy_out[:, 0] = E_hf_ref + xp.asarray(E_1B_delta) + xp.asarray(E_corr_delta)
        energy_out[:, 1] = E_hf_ref + xp.asarray(E_1B_delta)
        energy_out[:, 2] = xp.asarray(E_corr_delta)
        
        is_gpu_walker = hasattr(walkers.weight, 'device') or 'cupy' in str(type(walkers.weight))
        if is_gpu_walker and use_gpu:
            return cp.asarray(energy_out)
        return cp.asnumpy(energy_out) if use_gpu else energy_out

    return local_energy_single_det_uhf

# =============================================================================
# 5. MAIN INITIALIZATION, MPI MAPPING & PATCHING
# =============================================================================
comm = MPIHandler()
rank = comm.rank

if has_cupy:
    num_gpus = cp.cuda.runtime.getDeviceCount()
    if num_gpus > 0:
        device_id = rank % num_gpus
        cp.cuda.Device(device_id).use()
        if rank == 0:
            print(f">>> Found {num_gpus} GPUs. Mapping Rank {rank} to Device {device_id}")

if rank == 0:
    print(f">>> LOADING ML ASSETS...")

try:
    ml_model = load_model(MODEL_PATH, custom_objects={"log_cosh_loss": log_cosh_loss})
    X_scaler = joblib.load(os.path.join(DEPLOY_DIR, "X_scaler.save"))
    y_scaler = joblib.load(os.path.join(DEPLOY_DIR, "y_scaler.save"))
    P_hf_ref = np.load(os.path.join(DEPLOY_DIR, "P_hf_lowdin.npy"))
except Exception as e:
    if rank == 0:
        print(f"Error loading ML assets: {e}")
    sys.exit(1)

if rank == 0:
    mol = gto.M(atom=[("H", 0.74 * j, 0, 0) for j in range(N_ATOMS)], basis="sto-6g", verbose=0)
    mf = scf.UHF(mol)
    mf.kernel()
    
    gen_ipie_input_from_pyscf_chk(mf.chkfile, hamil_file=f"ham_h{N_ATOMS}.h5", wfn_file=f"wfn_h{N_ATOMS}.h5", verbose=0, chol_cut=1e-5)
    
    E_HF = mf.e_tot
    C_a = mf.mo_coeff[0] if np.ndim(mf.mo_coeff) == 3 else mf.mo_coeff
    _, S_sqrt, h_core, _ = get_dynamic_operators(mol)
else:
    E_HF = C_a = S_sqrt = h_core = None

E_HF = comm.comm.bcast(E_HF, root=0)
C_a = comm.comm.bcast(C_a, root=0)
S_sqrt = comm.comm.bcast(S_sqrt, root=0)
h_core = comm.comm.bcast(h_core, root=0)

afqmc = AFQMC.build_from_hdf5(
    num_elec=(N_ATOMS//2, N_ATOMS//2), ham_file=f"ham_h{N_ATOMS}.h5",
    wfn_file=f"wfn_h{N_ATOMS}.h5", num_walkers=WALKERS, num_blocks=NUM_BLOCKS,    
    num_steps_per_block=STEPS_PER_BLOCK, verbose=0, seed=TEST_SEED 
)

if has_cupy:
    afqmc.cuda = True

afqmc.mpi_handler = comm
if comm.size > 1:
    local_walkers = WALKERS // comm.size
    if rank < (WALKERS % comm.size): local_walkers += 1
    afqmc.nwalkers = local_walkers

ml_proxy = create_ml_local_energy_patch(
    ml_model, X_scaler, y_scaler, P_hf_ref, E_HF, S_sqrt, h_core, C_a, has_cupy, N_ATOMS
)

target_funcs = [
    "local_energy_single_det_uhf", 
    "local_energy_single_det_batch_gpu", 
    "local_energy_single_det_uhf_batch_gpu",
    "local_energy_single_det_uhf_batch"
]
targets = [getattr(ipie.estimators.local_energy_sd, f) for f in target_funcs if hasattr(ipie.estimators.local_energy_sd, f)]

for mod_name, module in list(sys.modules.items()):
    if module is None or not mod_name.startswith("ipie"): continue
    try:
        for attr_name, attr_value in vars(module).items():
            if attr_value in targets:
                setattr(module, attr_name, ml_proxy)
    except: pass

if hasattr(afqmc, 'propagator'):
    afqmc.propagator.local_energy = ml_proxy
if hasattr(afqmc, 'estimators'):
    try:
        afqmc.estimators['energy'].local_energy = ml_proxy
    except: pass

# =============================================================================
# 6. RUN WITH PROFILING
# =============================================================================
if rank == 0: 
    print("\n" + "#"*60 + "\n### STARTING ML-AFQMC PRODUCTION RUN ###\n" + "#"*60)
    start_time = time.perf_counter()

afqmc.run()

if rank == 0:
    end_time = time.perf_counter()
    
    # 1. True Peak Host RAM (in MB)
    peak_host_ram_mb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024.0
    
    # 2. True Peak GPU VRAM (TF + CuPy)
    tf_vram_mb = 0
    cp_vram_mb = 0
    current_device_id = rank % len(gpus) if gpus else 0
    
    if gpus:
        try:
            tf_vram_mb = tf.config.experimental.get_memory_info(f'GPU:{current_device_id}')['peak'] / (1024 ** 2)
        except Exception: pass
        
    if has_cupy:
        try:
            cp_vram_mb = cp.get_default_memory_pool().total_bytes() / (1024 ** 2)
        except AttributeError: pass
    
    total_peak_vram_mb = tf_vram_mb + cp_vram_mb

# =============================================================================
# 7. DATA EXTRACTION, ANALYSIS & SAVING
# =============================================================================
if rank == 0:
    print("\n" + "="*50)
    print(">>> PERFORMING AUTOCORRELATION ANALYSIS")
    print("="*50)
    
    final_energy = None
    final_error = None
    
    try:
        estimator_filename = afqmc.estimators.filename
        qmc_data = extract_observable(estimator_filename, "energy")
        y_energy = qmc_data["ETotal"][1:]
        df_ac = reblock_by_autocorr(y_energy, verbose=0)
        
        final_energy = float(df_ac["ETotal_ac"].iloc[0])
        final_error = float(df_ac["ETotal_error_ac"].iloc[0])
        print(f"  Final Reblocked Energy: {final_energy:.6f} Ha")
        print(f"  Final Reblocked Error:  {final_error:.6f} Ha")
        
    except Exception as e:
        print(f"  [Warning] Autocorrelation analysis failed: {e}")
        if 'y_energy' in locals() and len(y_energy) > 0:
            final_energy = float(np.mean(y_energy))
            final_error = float(np.std(y_energy) / np.sqrt(len(y_energy)))
            print(f"  Fallback Raw Mean Energy: {final_energy:.6f} ± {final_error:.6f} Ha")
    
    # -----------------------------
    # TIMING & FLOPs CALCULATION
    # -----------------------------
    total_time = end_time - start_time
    time_per_block = total_time / NUM_BLOCKS
    
    # Physics Overhead
    flops_physics_per_walker = 8 * (N_ATOMS ** 3) 
    
    # ML Proxy Overhead
    flops_nn_per_walker = estimate_nn_flops(ml_model)
    
    # Total Hardware Operations (Safely using global WALKERS and STEPS_PER_BLOCK)
    flops_total_per_step = (flops_physics_per_walker + flops_nn_per_walker) * WALKERS
    total_flops_run = flops_total_per_step * NUM_BLOCKS * STEPS_PER_BLOCK
    teraflops_per_sec = (total_flops_run / total_time) / 1e12

    metrics = {
        "system": SYSTEM_NAME,
        "n_atoms": N_ATOMS,
        "total_walkers": WALKERS,
        "num_blocks": NUM_BLOCKS,
        "results": {
            "final_energy_ha": final_energy,
            "final_error_ha": final_error
        },
        "timing": {
            "total_wall_time_sec": round(total_time, 4),
            "time_per_block_sec": round(time_per_block, 4)
        },
        "memory": {
            "peak_host_ram_mb": round(peak_host_ram_mb, 2),
            "peak_gpu_vram_mb": round(total_peak_vram_mb, 2),
            "gpu_vram_split": {
                "tensorflow_mb": round(tf_vram_mb, 2),
                "cupy_mb": round(cp_vram_mb, 2)
            }
        },
        "compute": {
            "flops_physics_per_walker": flops_physics_per_walker,
            "flops_nn_per_walker": flops_nn_per_walker,
            "total_tflops_executed": round(total_flops_run / 1e12, 6),
            "tflops_throughput": round(teraflops_per_sec, 6)
        }
    }

    print("\n" + "="*50)
    print(">>> ML PROXY SCALING METRICS SUMMARY")
    print("="*50)
    print(f"  Final Energy:         {final_energy} ± {final_error} Ha")
    print(f"  Total Wall Time:      {total_time:.4f} sec")
    print(f"  Peak Host RAM:        {peak_host_ram_mb:.2f} MB")
    print(f"  Peak GPU VRAM:        {total_peak_vram_mb:.2f} MB (TF: {tf_vram_mb:.1f} | CuPy: {cp_vram_mb:.1f})")
    print(f"  TFLOP/s Throughput:   {teraflops_per_sec:.6f}")
    print("="*50)

    save_path = f"ml_proxy_metrics_{SYSTEM_NAME}.json"
    with open(save_path, "w") as f:
        json.dump(metrics, f, indent=4)
    print(f"Metrics saved to {save_path}")

libnvvm.so successfully injected into process memory.


2026-02-28 19:40:08.987763: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TF memory growth enabled on 1 GPUs.
>>> Found 1 GPUs. Mapping Rank 0 to Device 0
>>> LOADING ML ASSETS...


I0000 00:00:1772325616.080224 1425449 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46712 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:41:00.0, compute capability: 8.6


# random seed is 999

############################################################
### STARTING ML-AFQMC PRODUCTION RUN ###
############################################################
            Block                   Weight            WeightFactor            HybridEnergy                  ENumer                  EDenom                  ETotal                  E1Body                  E2Body

[ML-PROXY] Intercepted call. Algebra executing on: GPU (CuPy/TF).
                0   2.0480000000000000e+03  2.0480000000000000e+03  0.0000000000000000e+00 -7.1709255792495343e+04  2.0480000000000000e+03 -3.5014285054929367e+01 -3.4978273034967295e+01 -3.6012019962072372e-02
                1   2.1435243963992098e+03  2.0480000000000000e+03 -1.8238236819074174e+01 -7.5069462429601626e+04  2.1435243963992098e+03 -3.5021510627873766e+01 -3.4978293128614638e+01 -4.3217499259135754e-02
                2   2.1435286792692668e+03  2.0480000000000000e+03 -1.8239154601167265e+01 -7.5085058621128177e+04 

In [1]:
(-35.147408 - -35.166309)*630

11.907629999999756